In [23]:
import os
import pandas as pd
import psycopg2

import pandas as pd
import psycopg2
from google import genai
from pgvector.psycopg2 import register_vector

PATH_EXAMPLE_CV = "./data/cv_example.pdf"
EMBEDDING_MODEL = "gemini-embedding-001"
LLM_MODEL_NAME = "gemini-2.5-flash-lite"
LLM_PROVIDER = "google_genai"
LLM_TEMPERATURE = 0.5
EMBED_BATCH_SIZE = 100
EMBED_CONCURRENCY = 5
EMBED_MAX_RETRIES = 3

DB_NAME = "market_fit"
DB_HOST = "localhost"
db_user = os.getenv("DB_USER")
db_pw = os.getenv("DB_PASSWORD")
api_key = os.getenv('GEMINI_API_KEY')
JOB_POSTINGS_TABLE = "job_postings"
HARD_SKILLS_TABLE = "hard_skills"
SOFT_SKILLS_TABLE = "soft_skills"
RESUMES_TABLE = "resumes"
INDUSTRIES_TABLE = "work_industries"
CANDIDATE_HARD_SKILLS_TABLE = "candidate_hard_skills"
CANDIDATE_SOFT_SKILLS_TABLE = "candidate_soft_skills"

In [ ]:
def db_connect() -> psycopg2.extensions.connection:
    conn = psycopg2.connect(
        host=DB_HOST, dbname=DB_NAME, user=db_user, password=db_pw
    )
    register_vector(conn)
    return conn

def filter_job_postings(industries: list) -> pd.DataFrame:
    filter_string = f"WHERE ai_industries::TEXT[] && ARRAY[{industries}]"
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT id, title, description FROM {JOB_POSTINGS_TABLE} {filter_string}")
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

def get_resume(resume_id: str) -> pd.DataFrame:
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT * FROM {RESUMES_TABLE} WHERE id = %s", (resume_id,))
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

In [ ]:
resume_df = get_resume('b6e8165a-b1af-4117-8a29-4a3a5fc95f32')
candidate_industries = resume_df["industries"][0]
matching_jobs_by_industries = filter_job_postings(candidate_industries)

In [ ]:
matching_jobs_by_industries


FULL QUERY: SELECT id, title, description FROM job_postings WHERE 1=1 AND (WHERE ai_industries::TEXT[] && ARRAY[['INFORMATION TECHNOLOGY & SERVICES', 'IT SERVICES AND IT CONSULTING', 'COMPUTER SOFTWARE']])


,id,title,description
0,2123414317,"Data Engineer (AWS, Redshift, Spark & SNS)",Sabemos que carreiras podem ser tão diferentes...
1,2122980329,Senior Data Engineer – GCP & Snowflake,Job Title: Senior Data Engineer – GCP & Snowfl...
2,2115963207,Data Engineer (SAP + Snowflake) - Remote Work,"At BairesDev®, we've been leading the way in t..."
3,2112042115,Senior Data Engineer (4627),Come work for a large global financial and ins...
4,2073491174,"Data Engineer (AWS, Redshift & Spark)",Sabemos que carreiras podem ser tão diferentes...
5,2043631830,Senior Data Engineer GCP,Job Description\n\nWe are looking for a Senior...
6,2039476314,Data & Analytics Engineer,Join BRP’s dynamic and inclusive Data & Analyt...
